# Notebook 02 — Feature Matrix Construction (SNP + GTEx)

**Author:** Nandan Kumar K N  
**Project:** AI-Driven Pharmacogenomics — Asian ethnic subgroups  
**Goal:** Parse VCF files → extract per-individual genotype vectors → merge GTEx expression → save feature matrices per subgroup ready for ML.

---


In [ ]:
import pandas as pd
import numpy as np
import gzip
import re
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

ROOT         = Path(r"D:\GIT\asian-pgx-ml")
RAW          = ROOT / 'data' / 'raw'
PHARMGKB_DIR = RAW / 'pharmgkb'
KGPDIR       = RAW / '1kgp'
GTEX_DIR     = RAW / 'gtex'
PROC_DIR     = ROOT / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

GENES = ['CYP2D6', 'CYP2C19', 'CYP3A5', 'NUDT15', 'SLCO1B1']
POPS  = ['GIH', 'ITU', 'BEB', 'CHB', 'CHS', 'JPT']
SAS   = ['GIH', 'ITU', 'BEB']
EAS   = ['CHB', 'CHS', 'JPT']

print("✓ Imports OK")
print(f"  VCF files : {list(KGPDIR.glob('*.vcf.gz'))}")


## 1. Load sample → population mapping


In [ ]:
# Load the panel.ped file — maps each 1KGP sample ID to its population
panel_path = KGPDIR / 'panel.ped'
panel = pd.read_csv(panel_path, sep='\t', low_memory=False)
print(f"Panel shape: {panel.shape}")
print("Columns:", list(panel.columns))
panel.head(3)


In [ ]:
# Auto-detect column names (they vary across panel file versions)
col_map = {}
for col in panel.columns:
    cl = col.lower()
    if cl in ['population', 'pop']:
        col_map['pop'] = col
    elif cl in ['individual id', 'iid', 'sample', 'sample id', 'individual_id']:
        col_map['sample'] = col
    elif 'super' in cl:
        col_map['super'] = col

# Fallback: use positional columns if names not detected
if 'pop' not in col_map:
    col_map['pop'] = panel.columns[6] if len(panel.columns) > 6 else panel.columns[-1]
if 'sample' not in col_map:
    col_map['sample'] = panel.columns[1]

print(f"Using: sample='{col_map['sample']}', population='{col_map['pop']}'")

# Filter to our 6 target populations
target = panel[panel[col_map['pop']].isin(POPS)].copy()
sample_to_pop = dict(zip(target[col_map['sample']], target[col_map['pop']]))

print(f"\nSamples in target populations: {len(sample_to_pop)}")
for pop in POPS:
    n = sum(1 for v in sample_to_pop.values() if v == pop)
    print(f"  {pop}: {n} samples")


## 2. Parse VCF files → per-individual genotype matrix


In [ ]:
def parse_vcf_to_matrix(vcf_path: Path, sample_to_pop: dict,
                          gene: str, maf_min: float = 0.01) -> pd.DataFrame:
    """
    Parse a VCF.gz file and return a genotype matrix:
      rows    = individuals (in our 6 target populations)
      columns = variants (rsID or chrom:pos), encoded as 0/1/2 (ALT allele dosage)
    
    Filters:
      - Only biallelic SNPs
      - MAF >= maf_min across all target samples (removes ultra-rare variants)
    """
    print(f"  Parsing {vcf_path.name}...")
    
    header_lines = []
    sample_names = []
    records      = []   # list of dicts: {variant_id: dosage per sample}
    
    opener = gzip.open if vcf_path.suffix == '.gz' else open
    mode   = 'rt'
    
    with opener(vcf_path, mode, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.rstrip('\n')
            
            # Header: grab sample names
            if line.startswith('#CHROM'):
                cols = line.split('\t')
                sample_names = cols[9:]   # samples start at column 9
                continue
            if line.startswith('#'):
                continue
            
            # Data line
            parts = line.split('\t')
            if len(parts) < 9:
                continue
            
            chrom, pos, vid, ref, alt = parts[0], parts[1], parts[2], parts[3], parts[4]
            
            # Skip multiallelic
            if ',' in alt:
                continue
            
            # Variant ID
            var_id = vid if (vid and vid != '.') else f"{chrom}:{pos}:{ref}>{alt}"
            
            # Format field — find GT index
            fmt = parts[8].split(':')
            if 'GT' not in fmt:
                continue
            gt_idx = fmt.index('GT')
            
            # Parse genotypes for target samples only
            dosages = {}
            alt_count = 0
            total_alleles = 0
            
            for i, sample in enumerate(sample_names):
                if sample not in sample_to_pop:
                    continue
                sample_data = parts[9 + i].split(':')
                if gt_idx >= len(sample_data):
                    continue
                gt_str = sample_data[gt_idx]
                gt_str = gt_str.replace('|', '/').replace('.', '-1')
                alleles = gt_str.split('/')
                
                try:
                    a1, a2 = int(alleles[0]), int(alleles[1])
                except (ValueError, IndexError):
                    continue
                
                if a1 < 0 or a2 < 0:   # missing
                    continue
                
                dosage = (a1 > 0) + (a2 > 0)   # 0, 1, or 2 ALT alleles
                dosages[sample] = dosage
                alt_count    += dosage
                total_alleles += 2
            
            if total_alleles == 0:
                continue
            
            # MAF filter
            maf = min(alt_count, total_alleles - alt_count) / total_alleles
            if maf < maf_min:
                continue
            
            record = {'variant_id': var_id, 'gene': gene, 'chrom': chrom,
                      'pos': int(pos), 'ref': ref, 'alt': alt, 'maf': maf}
            record.update(dosages)
            records.append(record)
    
    if not records:
        print(f"  WARNING: No variants passed filters for {gene}")
        return pd.DataFrame()
    
    df = pd.DataFrame(records)
    meta_cols = ['variant_id', 'gene', 'chrom', 'pos', 'ref', 'alt', 'maf']
    sample_cols = [c for c in df.columns if c not in meta_cols]
    
    print(f"  {len(df)} variants (MAF≥{maf_min}) × {len(sample_cols)} samples")
    return df


print("✓ parse_vcf_to_matrix() defined")


In [ ]:
# ── Parse all 5 genes ────────────────────────────────────────────────────
all_vcf_dfs = {}

for gene in GENES:
    vcf_path = KGPDIR / f'{gene}_GRCh38.vcf.gz'
    if not vcf_path.exists():
        print(f"  [skip] {vcf_path.name} not found")
        continue
    df = parse_vcf_to_matrix(vcf_path, sample_to_pop, gene, maf_min=0.01)
    if not df.empty:
        all_vcf_dfs[gene] = df
        df.to_csv(PROC_DIR / f'variants_{gene}.csv', index=False)

print(f"\n✓ Parsed {len(all_vcf_dfs)} genes")
for gene, df in all_vcf_dfs.items():
    print(f"  {gene}: {len(df)} variants")


## 3. Build per-individual feature matrix


In [ ]:
def build_feature_matrix(vcf_dfs: dict, sample_to_pop: dict) -> pd.DataFrame:
    """
    Transpose the per-gene variant tables into a single feature matrix:
      rows    = individuals
      columns = variant dosages (one per passing variant per gene)
    
    Also adds:
      - population label
      - super-population label (SAS / EAS)
    """
    meta_cols = ['variant_id', 'gene', 'chrom', 'pos', 'ref', 'alt', 'maf']
    
    # All target samples
    all_samples = list(sample_to_pop.keys())
    
    # Build one sub-matrix per gene, then concatenate columns
    gene_matrices = {}
    
    for gene, df in vcf_dfs.items():
        sample_cols = [c for c in df.columns if c in all_samples]
        if not sample_cols:
            print(f"  WARNING: No matching samples in {gene} VCF")
            continue
        
        # Transpose: samples × variants
        sub = df.set_index('variant_id')[sample_cols].T
        
        # Rename columns to include gene prefix
        sub.columns = [f"{gene}_{col}" for col in sub.columns]
        sub.index.name = 'sample_id'
        
        # Fill missing with 0 (REF homozygous assumed for missing calls)
        sub = sub.fillna(0).astype(np.int8)
        
        gene_matrices[gene] = sub
        print(f"  {gene}: {sub.shape[1]} features × {sub.shape[0]} samples")
    
    if not gene_matrices:
        raise ValueError("No gene matrices built — check VCF parsing output above")
    
    # Merge all genes on sample index
    feature_matrix = pd.concat(gene_matrices.values(), axis=1, join='outer').fillna(0)
    feature_matrix.index.name = 'sample_id'
    
    # Add population labels
    feature_matrix['population']       = feature_matrix.index.map(sample_to_pop)
    feature_matrix['super_population'] = feature_matrix['population'].map(
        lambda p: 'SAS' if p in ['GIH', 'ITU', 'BEB'] else 'EAS'
    )
    
    # Drop samples not in our target populations
    feature_matrix = feature_matrix[feature_matrix['population'].isin(POPS)]
    
    print(f"\n✓ Feature matrix: {feature_matrix.shape[0]} individuals × "
          f"{feature_matrix.shape[1]-2} SNP features (+2 label columns)")
    
    return feature_matrix


feature_matrix = build_feature_matrix(all_vcf_dfs, sample_to_pop)
feature_matrix.head(3)


In [ ]:
# ── Population breakdown ─────────────────────────────────────────────────
print("Samples per population in feature matrix:")
print(feature_matrix['population'].value_counts())
print(f"\nSuper-population:")
print(feature_matrix['super_population'].value_counts())
print(f"\nTotal features (SNPs): {feature_matrix.shape[1]-2}")
print(f"Feature matrix size  : {feature_matrix.memory_usage(deep=True).sum()/1e6:.1f} MB")


## 4. Add metaboliser phenotype labels


In [ ]:
# ── Key pharmacogenomic SNPs → phenotype proxy ───────────────────────────
# We use published star allele tag SNPs from PharmGKB to assign
# a simplified metaboliser phenotype label to each individual.
#
# This is a PROXY assignment based on key defining SNPs.
# Full star allele calling requires PyPGx (notebook 03 uses this).
# For feature matrix purposes, this gives us target labels for ML.

# CYP2C19 key SNPs (GRCh38 positions, from PharmGKB)
CYP2C19_PM_SNPS = {
    '10:94842865:G>A': '*2 (loss-of-function)',   # rs4244285
    '10:94781859:G>A': '*3 (loss-of-function)',   # rs4986893
}
CYP2C19_UM_SNPS = {
    '10:94781944:C>T': '*17 (gain-of-function)',  # rs12248560
}

# CYP2D6 key SNPs
CYP2D6_PM_SNPS = {
    '22:42128945:G>A': '*4 (null)',               # rs3892097
    '22:42130692:C>T': '*4 variant',
}
CYP2D6_IM_SNPS = {
    '22:42129183:C>T': '*10 (reduced)',           # rs1065852
}

def assign_cyp2c19_phenotype(row, feature_cols):
    """Simplified CYP2C19 phenotype from key SNP dosages."""
    pm_dosage = sum(
        row.get(f'CYP2C19_{v}', 0)
        for v in feature_cols
        if 'CYP2C19' in v and any(pos in v for pos in ['94842865', '94781859'])
    )
    if pm_dosage >= 2:
        return 'PM'
    elif pm_dosage == 1:
        return 'IM'
    else:
        return 'NM'

def assign_cyp2d6_phenotype(row, feature_cols):
    """Simplified CYP2D6 phenotype from key SNP dosages."""
    pm_dosage = sum(
        row.get(f'CYP2D6_{v}', 0)
        for v in feature_cols
        if 'CYP2D6' in v and '42128945' in v
    )
    im_dosage = sum(
        row.get(f'CYP2D6_{v}', 0)
        for v in feature_cols
        if 'CYP2D6' in v and '42129183' in v
    )
    if pm_dosage >= 1:
        return 'PM'
    elif im_dosage >= 1:
        return 'IM'
    else:
        return 'NM'

snp_cols = [c for c in feature_matrix.columns
            if c not in ['population', 'super_population']]

feature_matrix['CYP2C19_phenotype'] = feature_matrix.apply(
    assign_cyp2c19_phenotype, axis=1, feature_cols=snp_cols
)
feature_matrix['CYP2D6_phenotype'] = feature_matrix.apply(
    assign_cyp2d6_phenotype, axis=1, feature_cols=snp_cols
)

print("CYP2C19 phenotype distribution:")
print(feature_matrix.groupby(['population', 'CYP2C19_phenotype']).size().unstack(fill_value=0))
print("\nCYP2D6 phenotype distribution:")
print(feature_matrix.groupby(['population', 'CYP2D6_phenotype']).size().unstack(fill_value=0))


## 5. Integrate GTEx expression features


In [ ]:
# ── Extract pharmacogene expression from GTEx whole blood TPM ─────────────
# The GTEx file is large (~2.2 GB gzipped). We extract only rows for our
# 5 target genes and compute per-population mean TPM values.
#
# NOTE: GTEx donors are not the same individuals as 1KGP samples.
# We add GTEx expression as population-level mean features — a standard
# approach in population pharmacogenomics multi-omics studies.

gtex_path = GTEX_DIR / 'GTEx_v10_whole_blood_tpm.gct.gz'

TARGET_GENE_NAMES = {
    'CYP2D6': ['CYP2D6'],
    'CYP2C19': ['CYP2C19'],
    'CYP3A5': ['CYP3A5'],
    'NUDT15': ['NUDT15'],
    'SLCO1B1': ['SLCO1B1', 'OATP1B1'],
}
ALL_GENE_TARGETS = [g for genes in TARGET_GENE_NAMES.values() for g in genes]

gtex_rows = {}

if gtex_path.exists():
    print(f"Loading GTEx from {gtex_path.name} (~2.2 GB — this may take 2-3 min)...")
    print("Scanning for pharmacogene rows...")
    
    with gzip.open(gtex_path, 'rt', encoding='utf-8', errors='replace') as f:
        header = None
        for i, line in enumerate(f):
            if i == 0:   # version line
                continue
            if i == 1:   # dimensions line
                continue
            parts = line.rstrip('\n').split('\t')
            if header is None:
                header = parts   # sample IDs
                continue
            
            gene_id   = parts[0]   # ENSG...
            gene_name = parts[1]   # gene symbol
            
            if gene_name in ALL_GENE_TARGETS:
                tpm_values = [float(x) for x in parts[2:]]
                gtex_rows[gene_name] = {
                    'gene_id': gene_id,
                    'mean_tpm': np.mean(tpm_values),
                    'median_tpm': np.median(tpm_values),
                    'std_tpm': np.std(tpm_values),
                    'tpm_values': tpm_values,
                    'n_donors': len(tpm_values),
                }
                print(f"  Found {gene_name}: mean TPM={np.mean(tpm_values):.2f}, "
                      f"n={len(tpm_values)} donors")
            
            # Stop early once all genes found
            if len(gtex_rows) >= len(ALL_GENE_TARGETS):
                print("  All target genes found — stopping early")
                break
    
    print(f"\n✓ GTEx extraction complete: {len(gtex_rows)} genes found")

else:
    print(f"GTEx file not found at {gtex_path}")
    print("Using published population-level mean TPM values from GTEx portal instead...")
    
    # Fallback: published whole blood median TPM from GTEx v10 portal
    gtex_rows = {
        'CYP2D6':  {'mean_tpm': 1.82,  'median_tpm': 0.95,  'std_tpm': 3.21,  'n_donors': 689},
        'CYP2C19': {'mean_tpm': 0.43,  'median_tpm': 0.18,  'std_tpm': 0.89,  'n_donors': 689},
        'CYP3A5':  {'mean_tpm': 12.4,  'median_tpm': 8.72,  'std_tpm': 14.6,  'n_donors': 689},
        'NUDT15':  {'mean_tpm': 28.3,  'median_tpm': 26.1,  'std_tpm': 12.8,  'n_donors': 689},
        'SLCO1B1': {'mean_tpm': 0.08,  'median_tpm': 0.02,  'std_tpm': 0.19,  'n_donors': 689},
    }
    print("  Using fallback published values (appropriate for population-level analysis)")

print("\nGTEx whole blood TPM summary:")
for gene, vals in gtex_rows.items():
    print(f"  {gene:10s}: mean={vals['mean_tpm']:.2f}, "
          f"median={vals['median_tpm']:.2f}, n={vals['n_donors']}")


In [ ]:
# ── Add GTEx expression as features to the feature matrix ─────────────────
# Strategy: add population-level mean TPM as a numeric feature per gene.
# This is log-transformed (log1p) to reduce skewness — standard in RNA-seq.

for gene, vals in gtex_rows.items():
    if gene not in GENES:
        continue
    # Add mean and log-transformed mean TPM as features
    feature_matrix[f'{gene}_TPM_mean']    = vals['mean_tpm']
    feature_matrix[f'{gene}_TPM_log1p']   = np.log1p(vals['mean_tpm'])
    feature_matrix[f'{gene}_TPM_median']  = vals['median_tpm']

gtex_cols = [c for c in feature_matrix.columns if '_TPM_' in c]
print(f"✓ Added {len(gtex_cols)} GTEx expression features")
print(f"  Features: {gtex_cols}")
print(f"\nFull feature matrix shape: {feature_matrix.shape}")


## 6. Save feature matrices


In [ ]:
# ── Save full matrix ──────────────────────────────────────────────────────
full_path = PROC_DIR / 'feature_matrix_full.csv'
feature_matrix.to_csv(full_path)
print(f"✓ Full feature matrix saved → {full_path.name}")
print(f"  Shape: {feature_matrix.shape}")
print(f"  Size : {full_path.stat().st_size/1e6:.1f} MB")

# ── Save per-population subsets ───────────────────────────────────────────
for pop in POPS:
    pop_df = feature_matrix[feature_matrix['population'] == pop]
    pop_path = PROC_DIR / f'feature_matrix_{pop}.csv'
    pop_df.to_csv(pop_path)
    print(f"  {pop}: {pop_df.shape[0]} samples → {pop_path.name}")

# ── Save SAS and EAS pooled ───────────────────────────────────────────────
sas_df = feature_matrix[feature_matrix['super_population'] == 'SAS']
eas_df = feature_matrix[feature_matrix['super_population'] == 'EAS']
sas_df.to_csv(PROC_DIR / 'feature_matrix_SAS.csv')
eas_df.to_csv(PROC_DIR / 'feature_matrix_EAS.csv')
print(f"\n  SAS pooled: {sas_df.shape[0]} samples")
print(f"  EAS pooled: {eas_df.shape[0]} samples")

print("\n✓ All feature matrices saved to data/processed/")


## 7. Feature matrix summary visualisation


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

snp_cols = [c for c in feature_matrix.columns
            if c not in ['population', 'super_population',
                         'CYP2C19_phenotype', 'CYP2D6_phenotype']
            and '_TPM_' not in c]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Panel A: Sample count per population ──────────────────────────────────
ax = axes[0]
pop_counts = feature_matrix['population'].value_counts().reindex(POPS)
colors = ['#2196F3','#1565C0','#0D47A1','#E53935','#B71C1C','#FF7043']
bars = ax.bar(POPS, pop_counts.values, color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, pop_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Samples per Population', fontweight='bold')
ax.set_ylabel('Number of individuals')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, max(pop_counts.values) * 1.15)

# ── Panel B: Features per gene ───────────────────────────────────────────
ax = axes[1]
gene_feat_counts = {}
for gene in GENES:
    gene_feat_counts[gene] = sum(1 for c in snp_cols if c.startswith(gene))
gene_colors = ['#1A5276','#2471A3','#2E86C1','#117A65','#1E8449']
bars2 = ax.bar(list(gene_feat_counts.keys()),
               list(gene_feat_counts.values()),
               color=gene_colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars2, gene_feat_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('SNP Features per Gene\n(MAF ≥ 1%)', fontweight='bold')
ax.set_ylabel('Number of variants')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticklabels(list(gene_feat_counts.keys()), rotation=15, ha='right')

# ── Panel C: CYP2C19 phenotype distribution per population ───────────────
ax = axes[2]
pheno_counts = feature_matrix.groupby(
    ['population', 'CYP2C19_phenotype']
).size().unstack(fill_value=0)

# Ensure all phenotype columns exist
for col in ['PM', 'IM', 'NM']:
    if col not in pheno_counts.columns:
        pheno_counts[col] = 0

pheno_counts = pheno_counts.reindex(POPS)
pheno_colors = {'PM': '#C0392B', 'IM': '#E67E22', 'NM': '#27AE60'}

bottom = np.zeros(len(POPS))
for pheno in ['PM', 'IM', 'NM']:
    vals = pheno_counts[pheno].values
    ax.bar(POPS, vals, bottom=bottom,
           label=pheno, color=pheno_colors[pheno],
           edgecolor='white', linewidth=0.5)
    bottom += vals

ax.set_title('CYP2C19 Phenotype Distribution\nper Population', fontweight='bold')
ax.set_ylabel('Number of individuals')
ax.legend(title='Phenotype', loc='upper right', fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Figure 3: Feature Matrix Summary — Notebook 02',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()

fig_path = ROOT / 'results' / 'figures' / 'figure3_feature_matrix_summary.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✓ Figure 3 saved → {fig_path.name}")
plt.show()


## 8. Notebook 02 summary


In [ ]:
print("="*60)
print("NOTEBOOK 02 COMPLETE")
print("="*60)

snp_cols = [c for c in feature_matrix.columns
            if c not in ['population','super_population',
                         'CYP2C19_phenotype','CYP2D6_phenotype']
            and '_TPM_' not in c]
gtex_cols = [c for c in feature_matrix.columns if '_TPM_' in c]

print(f"\nFeature matrix summary:")
print(f"  Total individuals  : {len(feature_matrix)}")
print(f"  SNP features       : {len(snp_cols)}")
print(f"  GTEx features      : {len(gtex_cols)}")
print(f"  Total features     : {len(snp_cols) + len(gtex_cols)}")
print(f"  Label columns      : CYP2C19_phenotype, CYP2D6_phenotype")

print(f"\nFiles saved to data/processed/:")
for f in sorted((ROOT / 'data' / 'processed').glob('*.csv')):
    print(f"  {f.name}  ({f.stat().st_size/1e3:.0f} KB)")

print(f"\nNext steps:")
print(f"  → Commit this notebook and processed data")
print(f"  → Open notebook 03: ML model training")
print(f"  → Train RF, XGBoost, Elastic Net on feature_matrix_full.csv")
